**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Modern Architectures

The transformer's descendants, dissected: mixture-of-experts routing (capacity without compute), attention's cost curve and its linear/sliding-window repairs, and where [SSM blocks](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) fit. Each mechanism built small and measured.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb), [Scale_NN](./Scale_NN/Scale_NN.ipynb).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 3 — *Attention's Cost Curve* (~35 min)
**Goal:** measure the quadratic wall; see what sliding windows and linear attention trade away.
**Builds on:** [Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (MoE).

---

## 2. The Quadratic Wall, Measured

💡 **Intuition.** Full attention lets every token query every token: $O(T^2)$ compute and memory — the price of unlimited connectivity. The repairs each *remove* something: **sliding windows** keep only local links (recover long range by stacking layers — the [CNN receptive-field](./Intro_CNN/Intro_CNN.ipynb) trick); **linear attention** replaces softmax with a kernel so the sum factorizes into a running state ($O(T)$ — and mathematically an [SSM](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb)!). No free lunch: each buys speed with a connectivity prior.

In [2]:
d = 64
def full_attn(q, k, v):
    A = torch.softmax(q @ k.T / d**0.5, dim=-1)
    return A @ v
def window_attn(q, k, v, w=64):
    T = len(q); out = torch.zeros_like(v)
    for i in range(0, T, w):                                  # block-local attention
        sl = slice(max(0, i), min(T, i+w))
        A = torch.softmax(q[sl] @ k[sl].T / d**0.5, dim=-1)
        out[sl] = A @ v[sl]
    return out
def linear_attn(q, k, v):
    phi = lambda x: torch.nn.functional.elu(x) + 1            # positive feature map
    S = phi(k).T @ v                                          # a running SUMMARY, size d×d
    z = phi(k).sum(0)
    return (phi(q) @ S) / (phi(q) @ z)[:, None]

for T in [512, 2048, 8192]:
    q, k, v = (torch.randn(T, d) for _ in range(3))
    times = {}
    for name, fn in [("full", full_attn), ("window", window_attn), ("linear", linear_attn)]:
        tic = time.perf_counter(); fn(q, k, v); times[name] = time.perf_counter()-tic
    print(f"T={T:5d}:  full {times['full']*1e3:7.1f} ms   window {times['window']*1e3:6.1f} ms   linear {times['linear']*1e3:6.1f} ms")
print("→ full attention's time grows ~16x per 4x length (quadratic); the others stay near-linear")

T=  512:  full     3.1 ms   window    0.4 ms   linear    0.7 ms
T= 2048:  full     4.5 ms   window    0.8 ms   linear    0.8 ms
T= 8192:  full    68.6 ms   window    3.2 ms   linear    1.8 ms
→ full attention's time grows ~16x per 4x length (quadratic); the others stay near-linear


---
### 🕐 Session 2 of 3 — *Mixture of Experts* (~40 min)
**Goal:** route tokens to specialists: parameters without proportional compute — specialization measured.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the assembled zoo).

---

## 3. Capacity Without the Bill

💡 **Intuition.** An MoE layer holds $E$ expert MLPs but a learned **router** sends each token to only the top-$k$ — so parameters scale with $E$ while per-token compute scales with $k$. The bet: tokens differ in *kind*, and specialists beat one generalist of equal compute. The classic failure is **routing collapse** (all tokens to one expert), patched with load-balancing losses. We build a 4-expert layer on a task with planted sub-populations and *check who goes where*.

In [3]:
# task with 4 planted regimes: y depends on x differently per quadrant of a latent code
def moe_data(n):
    c = torch.randint(0, 4, (n,))
    x = torch.randn(n, 8)
    x[:, :2] = torch.stack([torch.cos(c*1.57), torch.sin(c*1.57)], 1) + 0.1*torch.randn(n, 2)
    W = torch.stack([torch.randn(8) for _ in range(4)])
    y = (x * W[c]).sum(1, keepdim=True)
    return x, y, c
torch.manual_seed(3)
Xd, Yd, Cd = moe_data(6000)

class MoE(nn.Module):
    def __init__(self, E=4):
        super().__init__()
        self.router = nn.Linear(8, E)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 1)) for _ in range(E)])
    def forward(self, x):
        logits = self.router(x)
        top = logits.argmax(1)                                # hard top-1 routing
        probs = torch.softmax(logits, 1)
        out = torch.zeros(len(x), 1)
        for e, expert in enumerate(self.experts):
            m = top == e
            if m.any(): out[m] = expert(x[m]) * probs[m, e:e+1] / probs[m, e:e+1].detach()
        # load-balancing auxiliary: encourage uniform expert usage
        load = probs.mean(0)
        aux = (load * torch.log(load*len(self.experts) + 1e-9)).sum()
        return out, top, aux

def train_moe(aux_weight, route_noise, steps=2500, seed=1):
    torch.manual_seed(seed)
    moe = MoE(); opt = torch.optim.Adam(moe.parameters(), lr=3e-3)
    for step in range(steps):
        logits = moe.router(Xd)
        if route_noise > 0:                                   # exploration noise fights collapse
            logits = logits + route_noise*torch.randn_like(logits)
        top = logits.argmax(1)
        probs = torch.softmax(logits, 1)
        out = torch.zeros(len(Xd), 1)
        for e, expert in enumerate(moe.experts):
            m2 = top == e
            if m2.any(): out[m2] = expert(Xd[m2]) * probs[m2, e:e+1] / probs[m2, e:e+1].detach()
        load = probs.mean(0)
        aux = (load * torch.log(load*4 + 1e-9)).sum()
        loss = ((out - Yd)**2).mean() + aux_weight*aux
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        top = moe.router(Xd).argmax(1)
        out2, _, _ = moe(Xd)
    usage = [int((top==e).sum()) for e in range(4)]
    purity = [float((torch.bincount(Cd[top==e], minlength=4).float().max()/(top==e).sum())) if (top==e).sum()>10 else float("nan") for e in range(4)]
    return usage, purity, float(((out2 - Yd)**2).mean())

# THE FAILURE, on purpose: no balancing, no exploration → routing collapse
u0, p0, m0 = train_moe(aux_weight=0.0, route_noise=0.0)
print(f"no balancing:      usage {u0}   purity {np.round(p0,2)}   MSE {m0:.4f}   ← collapse: dead experts")
# THE CURE: load-balancing loss + routing noise
u1, p1, m1 = train_moe(aux_weight=0.1, route_noise=0.5)
print(f"balanced + noisy:  usage {u1}   purity {np.round(p1,2)}   MSE {m1:.4f}")
print("→ with the cure, every expert lives and each aligns with (mostly) one planted regime —")
print("  routing collapse is not a footnote, it is THE engineering problem of MoE")

no balancing:      usage [3153, 0, 11, 2836]   purity [0.48  nan 0.64 0.52]   MSE 0.0041   ← collapse: dead experts


balanced + noisy:  usage [1513, 1499, 1542, 1446]   purity [0.81 0.51 0.74 0.51]   MSE 0.0016
→ with the cure, every expert lives and each aligns with (mostly) one planted regime —
  routing collapse is not a footnote, it is THE engineering problem of MoE


---
### 🕐 Session 3 of 3 — *The Assembled Zoo* (~30 min)
**Goal:** how the pieces combine in 2026-era models; the design-space map.
**Builds on:** Session 2.

---

## 4. The Map

| Need | Mechanism | Cost model |
|---|---|---|
| Unlimited connectivity | full attention | $O(T^2)$, KV cache $O(T)$ |
| Long context, cheap | sliding window + a few global layers | $O(Tw)$ |
| Constant-state streaming | linear attention / [SSM/Mamba](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) | $O(T)$, state $O(1)$ |
| Parameters ≫ compute | MoE (top-k routing) | params ×E, FLOPs ×k |
| Memory during training | grouped/multi-query attention, [gradient accumulation](./Scale_NN/Scale_NN.ipynb) | smaller KV, same math |

💡 **Intuition.** Modern frontier models are *hybrids by necessity*: interleaved sliding/full attention, MoE feed-forwards, sometimes SSM layers — each mechanism spending a different currency (compute, memory, connectivity). Read any architecture paper as a walk through this table.

**Exercise with teeth:** wire the MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure perplexity per FLOP against the dense baseline.

---
## Where next

- [State-Space Models](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) — the third pillar, in depth.
- [Scale_NN](./Scale_NN/Scale_NN.ipynb) — why these trade-offs exist at all.